In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm
import os
import matplotlib.ticker as ticker
import matplotlib as mpl

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
def find_corners(img): 
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # 查找棋盘格角点;
    pattern_size = (6, 8)
    ret, corners = cv2.findChessboardCorners(gray, pattern_size,None)
    #print(ret)
    if ret:
        # 精细查找角点
        corners2 = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
        # 显示角点
        #cv2.drawChessboardCorners(img, pattern_size, corners2, ret)
    return ret, corners

In [4]:
def sortpoints(arr):
    new_pts = np.round(arr/10)*10
    arrSortIndex = np.lexsort((new_pts[:, 1], new_pts[:, 0]))
    newArr = []
    for num in arrSortIndex:
        newArr.append(list(arr[num]))
    return np.array(newArr)

In [5]:
def GetHomography(frame_left,frame_right,again):
    #frame_right = cv2.resize(frame_right,(frame_left.shape[1],frame_left.shape[0]))
    ret1, src_pts = find_corners(frame_left)
    ret2, dst_pts = find_corners(frame_right)
    
    if ret1==True and ret2==True:
        if again == 0:
            src_pts = src_pts[:,0]
        if again == 1:
            src_pts = src_pts[:,0][::-1]
        dst_pts = dst_pts[:,0]
        #src_pts = sortpoints(src_pts)
        #dst_pts = sortpoints(dst_pts)
        H=cv2.findHomography(src_pts,dst_pts,cv2.RANSAC,5)
        H_matrix=H[0] 
        return src_pts,dst_pts,H_matrix
    else:      
        print("No corners")
        return 0,0,0

In [6]:
def GetHomography_mid(frame_left,frame_right,again):
    #frame_right = cv2.resize(frame_right,(frame_left.shape[1],frame_left.shape[0]))
    ret1, src_pts = find_corners(frame_left)
    ret2, dst_pts = find_corners(frame_right)
    
    if ret1==True and ret2==True:
        if again == 0:
            src_pts = src_pts[:,0]
        if again == 1:
            src_pts = src_pts[:,0][::-1]
        dst_pts = dst_pts[:,0]
        mid_pts = (src_pts+dst_pts)/2
        #src_pts = sortpoints(src_pts)
        #dst_pts = sortpoints(dst_pts)
        H_left=cv2.findHomography(src_pts,mid_pts,cv2.RANSAC,5)
        H_right=cv2.findHomography(dst_pts,mid_pts,cv2.RANSAC,5)
        H_matrix_left=H_left[0] 
        H_matrix_right=H_right[0] 
        return src_pts,dst_pts,H_matrix_left,H_matrix_right
    else:      
        print("No corners")
        return 0,0,0

In [7]:
def image_stitching(frame_left,frame_right,H_matrix):
    h,w=frame_left.shape[:2]
    h1,w1=frame_right.shape[:2]
    shft=np.array([[1.0,0,w],[0,1.0,0],[0,0,1.0]])
    M=np.dot(shft,H_matrix)            #Get the projection mapping relationship from the left image to the right image
    dst_corners=cv2.warpPerspective(frame_left,M,(w*2,h))#Perspective transformation, new image can fit two full images
    dst_corners[0:h,w:int(w+np.ceil(M[0][2]))] = 0.5*frame_right[0:h,:int(np.ceil(M[0][2]))] + 0.5*dst_corners[0:h,w:int(w+np.ceil(M[0][2]))]
    dst_corners[0:h,int(w+np.ceil(M[0][2])):2*w] = frame_right[0:h,int(np.ceil(M[0][2])):w]
    #stitched = dst_corners[: , int(np.around(M[0][2])): -1]
    #print(int(np.around(M[0][2])))
    stitched = dst_corners[: , 1800: -1]
    return stitched

In [8]:
def image_stitching_mid(frame_left,frame_right,H_matrix_left,H_matrix_right):
    h,w=frame_left.shape[:2]
    h1,w1=frame_right.shape[:2]
    shft=np.array([[1.0,0,w],[0,1.0,0],[0,0,1.0]])
    M=np.dot(shft,H_matrix_left)            #Get the projection mapping relationship from the left image to the right image
    dst_corners=cv2.warpPerspective(frame_left,M,(w*2,h))#Perspective transformation, new image can fit two full images
    frame_right_homo=cv2.warpPerspective(frame_right,H_matrix_right,(w1,h1))
    overlap_width = int(np.ceil(M[0][2]))
    for c in range(3):  # 对于RGB的每个通道
        left_overlap = dst_corners[0:h, w:w + overlap_width, c]
        right_overlap = frame_right_homo[0:h, :overlap_width, c]
        mask = right_overlap > 0
        left_overlap[mask] = 0.5 * right_overlap[mask] + 0.5 * left_overlap[mask]
        # 如果右图的像素是黑色，则保持左图像素不变
    dst_corners[0:h, w:w + overlap_width] = dst_corners[0:h, w:w + overlap_width]
    dst_corners[0:h,int(w+np.ceil(M[0][2])):2*w] = frame_right_homo[0:h,int(np.ceil(M[0][2])):w]
    stitched = dst_corners[: , 1800: -1]
    return stitched

In [17]:
def image_stitching_mid_no55(frame_left,frame_right,H_matrix_left,H_matrix_right):
    h,w=frame_left.shape[:2]
    h1,w1=frame_right.shape[:2]
    shft=np.array([[1.0,0,w],[0,1.0,0],[0,0,1.0]])
    M=np.dot(shft,H_matrix_left)            #Get the projection mapping relationship from the left image to the right image
    dst_corners=cv2.warpPerspective(frame_left,M,(w*2,h))#Perspective transformation, new image can fit two full images
    frame_right_homo=cv2.warpPerspective(frame_right,H_matrix_right,(w1,h1))
    overlap_width = int(np.ceil(M[0][2]))
    for c in range(3):  # 对于RGB的每个通道
        left_overlap = dst_corners[0:h, w:w + overlap_width, c]
        right_overlap = frame_right_homo[0:h, :overlap_width, c]
        mask = right_overlap > 0
        left_overlap[mask] = right_overlap[mask] + left_overlap[mask]
        left_overlap[mask] = np.clip(left_overlap[mask], 0, 255)
        # 如果右图的像素是黑色，则保持左图像素不变
    dst_corners[0:h, w:w + overlap_width] = dst_corners[0:h, w:w + overlap_width]
    dst_corners[0:h,int(w+np.ceil(M[0][2])):2*w] = frame_right_homo[0:h,int(np.ceil(M[0][2])):w]
    stitched = dst_corners[: , 1800: -1]
    return stitched

In [9]:
def apply_radial_blur_ellipse(img, blur_strength=30, ellipse_scale=0.5):
    # 读取图像
    if img is None:
        print("Image not found, please check the path.")
        return None

    # 获取图像尺寸并计算中心点
    height, width = img.shape[:2]
    center_x, center_y = width // 2, height // 2

    # 创建椭圆形径向渐变遮罩
    # 椭圆形状遮罩在x和y方向上的扩展由ellipse_scale参数控制
    x = np.arange(width)
    y = np.arange(height).reshape(height, 1)
    # Making the horizontal axis of the ellipse longer
    d = np.sqrt(((x - center_x) / ellipse_scale)**2 + ((y - center_y)/ ellipse_scale)**2)
    sigma = (width**2 + height**2)**0.5 / blur_strength  # 模糊强度控制
    mask = np.exp(-(d**2) / (2 * sigma**2))
    mask = (mask - mask.min()) / (mask.max() - mask.min())
    mask = mask.astype(np.float32)
    
    # 将单通道遮罩扩展为三通道
    mask = np.stack((mask,)*3, axis=-1)
    
    # 创建模糊图像
    blurred = cv2.GaussianBlur(img, (0, 0), sigmaX=sigma, sigmaY=sigma)

    # 将遮罩应用于模糊图像与原始图像
    img_blur = cv2.convertScaleAbs(blurred * (1 - mask) + img * mask)
    
    return img_blur

In [10]:
def draw_cross(image, cross_radius=20):
    """
    在传入的图像中绘制十字架
    :param image: 传入的需要绘制十字架的图片
    :param cross_radius: 需要绘制的十字架的半径
    :return: 返回绘制好十字架的图像
    """
    center_x = image.shape[1] // 2  # 图像的中心点x坐标
    center_y = image.shape[0] // 2
    cross_x1 = center_x - cross_radius  # 十字的左顶点
    cross_x2 = center_x + cross_radius
    cross_y1 = center_y - cross_radius  # 十字的上顶点
    cross_y2 = center_y + cross_radius
    # 使用cv2.polylines()画多条直线也可以用来画多边形
    line1 = np.array([[cross_x1, center_y], [cross_x2, center_y]], np.int32).reshape((-1, 1, 2))
    line2 = np.array([[center_x, cross_y1], [center_x, cross_y2]], np.int32).reshape((-1, 1, 2))
    # line1 = np.array([[100, 100], [300, 100]], np.int32).reshape((-1, 1, 2))
    
    cv2.polylines(image, pts=[line1, line2], isClosed=False, color=(0, 255, 0), thickness=2, lineType=cv2.LINE_8)
    return image

In [11]:
def remove_ds_store(dir_path):
    for root, dirs, files in os.walk(dir_path):
        for file in files:
            if file == '.DS_Store':
                os.remove(os.path.join(root, file))
                print(f"Removed: {os.path.join(root, file)}")

In [12]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import DotProduct, WhiteKernel
from sklearn.gaussian_process.kernels import ConstantKernel, RBF
def gaussian_process(H11_entry,sigma_n):
    X = range(len(H11_entry))
    y = H11_entry
    X = np.array(X).reshape(-1, 1)
    y = np.array(y).reshape(-1, 1)

    l = 0.1
    sigma_f = 2
    kernel = ConstantKernel(constant_value=sigma_f, constant_value_bounds=(1e-2, 1e2)) \
                * RBF(length_scale=l, length_scale_bounds=(1e-2, 1e2))
    gpr = GaussianProcessRegressor(kernel=kernel, alpha=sigma_n**2, n_restarts_optimizer=10).fit(X, y)
    y_pred = gpr.predict(X, return_std=True)[0]
    y_pred = y_pred.reshape([y_pred.shape[0]])
    return y_pred

In [13]:
dir_path = r'/Users/Administrator/Desktop/research/video stitching/frames'
remove_ds_store(dir_path)

In [14]:
normal_root_left = r'/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024'
normal_root_right = r'/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024'
STR_root_left = r'/Users/Administrator/Desktop/research/video stitching/frames/STR left 2024'
STR_root_right = r'/Users/Administrator/Desktop/research/video stitching/frames/STR right 2024'
all_img_normal_left = os.listdir(normal_root_left)
all_img_normal_right = os.listdir(normal_root_right)
all_img_STR_left = os.listdir(STR_root_left)
all_img_STR_right = os.listdir(STR_root_right)
all_img_normal_left.sort(key= lambda x:int(x[10:-4]))
all_img_normal_right.sort(key= lambda x:int(x[10:-4]))
all_img_STR_left.sort(key= lambda x:int(x[10:-4]))
all_img_STR_right.sort(key= lambda x:int(x[10:-4]))
each_H_matrix_left = []
each_H_matrix_right = []
for i in range(len(all_img_STR_right)):
    frame_left = cv2.imread(normal_root_left + '/' + all_img_normal_left[i])
    frame_right = cv2.imread(normal_root_right + '/' + all_img_normal_right[i])
    print(i)
    src_pts,dst_pts,H_matrix_left,H_matrix_right = GetHomography_mid(frame_left,frame_right,0)

    if abs(H_matrix_left[0,0]-1) > 0.5 or abs(H_matrix_right[0,0]-1) > 0.5:
        src_pts,dst_pts,H_matrix_left,H_matrix_right = GetHomography_mid(frame_left,frame_right,1)
    each_H_matrix_left.append(H_matrix_left)
    each_H_matrix_right.append(H_matrix_right)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

In [15]:
left_H11_entry = []
left_H12_entry = []
left_H13_entry = []
left_H21_entry = []
left_H22_entry = []
left_H23_entry = []
left_H31_entry = []
left_H32_entry = []
for j in range(500):
    left_H11_entry.append(each_H_matrix_left[j][0][0])
    left_H12_entry.append(each_H_matrix_left[j][0][1])
    left_H13_entry.append(each_H_matrix_left[j][0][2])
    left_H21_entry.append(each_H_matrix_left[j][1][0])
    left_H22_entry.append(each_H_matrix_left[j][1][1])
    left_H23_entry.append(each_H_matrix_left[j][1][2])
    left_H31_entry.append(each_H_matrix_left[j][2][0])
    left_H32_entry.append(each_H_matrix_left[j][2][1])
right_H11_entry = []
right_H12_entry = []
right_H13_entry = []
right_H21_entry = []
right_H22_entry = []
right_H23_entry = []
right_H31_entry = []
right_H32_entry = []
for j in range(500):
    right_H11_entry.append(each_H_matrix_right[j][0][0])
    right_H12_entry.append(each_H_matrix_right[j][0][1])
    right_H13_entry.append(each_H_matrix_right[j][0][2])
    right_H21_entry.append(each_H_matrix_right[j][1][0])
    right_H22_entry.append(each_H_matrix_right[j][1][1])
    right_H23_entry.append(each_H_matrix_right[j][1][2])
    right_H31_entry.append(each_H_matrix_right[j][2][0])
    right_H32_entry.append(each_H_matrix_right[j][2][1])

In [19]:
leftG_H_11=gaussian_process(left_H11_entry,0.01)
leftG_H_12=gaussian_process(left_H12_entry,0.01)
leftG_H_13=gaussian_process(left_H13_entry,10)
leftG_H_21=gaussian_process(left_H21_entry,0.01)
leftG_H_22=gaussian_process(left_H22_entry,0.01)
leftG_H_23=gaussian_process(left_H23_entry,10)
leftG_H_31=gaussian_process(left_H31_entry,0.00001)
leftG_H_32=gaussian_process(left_H32_entry,0.00001)
rightG_H_11=gaussian_process(right_H11_entry,0.01)
rightG_H_12=gaussian_process(right_H12_entry,0.01)
rightG_H_13=gaussian_process(right_H13_entry,10)
rightG_H_21=gaussian_process(right_H21_entry,0.01)
rightG_H_22=gaussian_process(right_H22_entry,0.01)
rightG_H_23=gaussian_process(right_H23_entry,10)
rightG_H_31=gaussian_process(right_H31_entry,0.00001)
rightG_H_32=gaussian_process(right_H32_entry,0.00001)
for i in range(len(all_img_STR_right)):
    frame_left = cv2.imread(STR_root_left + '/' + all_img_STR_left[i])
    frame_right = cv2.imread(STR_root_right + '/' + all_img_STR_right[i])
    print(i)
    H_matrix_left = np.array([[leftG_H_11[i],leftG_H_12[i],leftG_H_13[i]],[leftG_H_21[i],leftG_H_22[i],leftG_H_23[i]],[leftG_H_31[i],leftG_H_32[i],1]])
    H_matrix_right = np.array([[rightG_H_11[i],rightG_H_12[i],rightG_H_13[i]],[rightG_H_21[i],rightG_H_22[i],rightG_H_23[i]],[rightG_H_31[i],rightG_H_32[i],1]])
    fusion_img = image_stitching_mid(frame_left,frame_right,H_matrix_left,H_matrix_right)
    fusion_img = apply_radial_blur_ellipse(fusion_img, blur_strength=30, ellipse_scale=6)
    #fusion_img = draw_cross(fusion_img, cross_radius=20)
    cv2.imwrite(r'/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024 no cross/{}.jpg'.format(i), fusion_img)  # 存储为图像

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

In [81]:
img_root = r'/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024'
fps = 33    #FPS
framesize = cv2.imread(img_root + '/' + '0.jpg')

size=(framesize.shape[1],framesize.shape[0])
fourcc = cv2.VideoWriter_fourcc(*'MJPG')
videoWriter = cv2.VideoWriter(r'/Users/Administrator/Desktop/research/video stitching/video/STR video 2024 blur.avi',fourcc,fps,size,True)

all_img_file = os.listdir(img_root)
all_img_file.sort(key= lambda x:int(x[0:-4]))

for img_file in all_img_file:
    frame = cv2.imread(img_root + '/' + img_file)
    videoWriter.write(frame)
    print(img_root + '/' + img_file + ' done!')
videoWriter.release()

/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/0.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/1.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/2.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/3.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/4.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/5.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/6.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/7.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/8.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/9.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/10.jpg done!
/Users/Ad

/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/95.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/96.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/97.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/98.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/99.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/100.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/101.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/102.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/103.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/104.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/105.jpg

/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/190.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/191.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/192.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/193.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/194.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/195.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/196.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/197.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/198.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/199.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/20

/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/284.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/285.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/286.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/287.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/288.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/289.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/290.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/291.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/292.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/293.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/29

/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/377.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/378.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/379.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/380.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/381.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/382.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/383.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/384.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/385.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/386.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/38

/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/467.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/468.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/469.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/470.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/471.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/472.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/473.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/474.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/475.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/476.jpg done!
/Users/Administrator/Desktop/research/video stitching/STR stitched image 2024/47

In [78]:
for i in range(len(all_img_STR_right)):
    frame_left = cv2.imread(normal_root_left + '/' + all_img_normal_left[i])
    frame_right = cv2.imread(normal_root_right + '/' + all_img_normal_right[i])
    print(i)
    src_pts,dst_pts,H_matrix_left,H_matrix_right = GetHomography_mid(frame_left,frame_right,0)
    
    if abs(H_matrix_left[0,0]-1) > 0.5 or abs(H_matrix_right[0,0]-1) > 0.5:
        src_pts,dst_pts,H_matrix_left,H_matrix_right = GetHomography_mid(frame_left,frame_right,1)
    fusion_img = image_stitching_mid(frame_left,frame_right,H_matrix_left,H_matrix_right)
    fusion_img = apply_radial_blur_ellipse(fusion_img, blur_strength=30, ellipse_scale=6)
    fusion_img = draw_cross(fusion_img, cross_radius=20)
    cv2.imwrite(r'/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/{}.jpg'.format(i), fusion_img)  # 存储为图像

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

In [79]:
img_root = r'/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024'
fps = 33    #FPS
framesize = cv2.imread(img_root + '/' + '0.jpg')

size=(framesize.shape[1],framesize.shape[0])
fourcc = cv2.VideoWriter_fourcc(*'MJPG')
videoWriter = cv2.VideoWriter(r'/Users/Administrator/Desktop/research/video stitching/video/normal video 2024.avi',fourcc,fps,size,True)

all_img_file = os.listdir(img_root)
all_img_file.sort(key= lambda x:int(x[0:-4]))

for img_file in all_img_file:
    frame = cv2.imread(img_root + '/' + img_file)
    videoWriter.write(frame)
    print(img_root + '/' + img_file + ' done!')
videoWriter.release()

/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/0.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/1.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/2.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/3.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/4.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/5.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/6.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/7.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/8.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/9.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched 

/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/93.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/94.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/95.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/96.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/97.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/98.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/99.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/100.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/101.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/102.jpg done!
/Users/Administrator/Desktop/research/video stitching/nor

/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/182.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/183.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/184.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/185.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/186.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/187.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/188.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/189.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/190.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/191.jpg done!
/Users/Administrator/Desktop/research/video stitch

/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/273.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/274.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/275.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/276.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/277.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/278.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/279.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/280.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/281.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/282.jpg done!
/Users/Administrator/Desktop/research/video stitch

/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/361.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/362.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/363.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/364.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/365.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/366.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/367.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/368.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/369.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/370.jpg done!
/Users/Administrator/Desktop/research/video stitch

/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/451.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/452.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/453.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/454.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/455.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/456.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/457.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/458.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/459.jpg done!
/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024/460.jpg done!
/Users/Administrator/Desktop/research/video stitch

In [74]:
i= 7
frame_left = cv2.imread(normal_root_left + '/' + all_img_normal_left[i])
frame_right = cv2.imread(normal_root_right + '/' + all_img_normal_right[i])
print(i)
src_pts,dst_pts,H_matrix_left,H_matrix_right = GetHomography_mid(frame_left,frame_right,0)

if abs(H_matrix_left[0,0]-1) > 0.5 or abs(H_matrix_right[0,0]-1) > 0.5:
    src_pts,dst_pts,H_matrix_left,H_matrix_right = GetHomography_mid(frame_left,frame_right,1)
fusion_img = image_stitching_mid(frame_left,frame_right,H_matrix_left,H_matrix_right)
fusion_img = apply_radial_blur_ellipse(fusion_img, blur_strength=30, ellipse_scale=6)

1


In [5]:
normal_root_left = r'/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024'

img_root = r'/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024'
fps = 33    #FPS
framesize = cv2.imread(normal_root_left  + '/' + 'chessboard0.jpg')

size=(framesize.shape[1],framesize.shape[0])
fourcc = cv2.VideoWriter_fourcc(*'MJPG')
videoWriter = cv2.VideoWriter(r'/Users/Administrator/Desktop/research/video stitching/video/normal video left 2024.avi',fourcc,fps,size,True)

all_img_file = os.listdir(normal_root_left)
all_img_file.sort(key= lambda x:int(x[10:-4]))

for img_file in all_img_file:
    frame = cv2.imread(normal_root_left + '/' + img_file)
    videoWriter.write(frame)
    print(normal_root_left + '/' + img_file + ' done!')
videoWriter.release()

/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard0.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard1.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard2.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard3.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard4.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard5.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard6.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard7.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard8.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR

/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard80.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard81.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard82.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard83.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard84.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard85.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard86.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard87.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard88.jpg done!
/Users/Administrator/Desktop/research/video stitching/f

/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard160.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard161.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard162.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard163.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard164.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard165.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard166.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard167.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard168.jpg done!
/Users/Administrator/Desktop/research/video st

/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard240.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard241.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard242.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard243.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard244.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard245.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard246.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard247.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard248.jpg done!
/Users/Administrator/Desktop/research/video st

/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard320.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard321.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard322.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard323.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard324.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard325.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard326.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard327.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard328.jpg done!
/Users/Administrator/Desktop/research/video st

/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard400.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard401.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard402.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard403.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard404.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard405.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard406.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard407.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard408.jpg done!
/Users/Administrator/Desktop/research/video st

/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard479.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard480.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard481.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard482.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard483.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard484.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard485.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard486.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal left 2024/chessboard487.jpg done!
/Users/Administrator/Desktop/research/video st

In [6]:
normal_root_right = r'/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024'

img_root = r'/Users/Administrator/Desktop/research/video stitching/normal stitched image 2024'
fps = 33    #FPS
framesize = cv2.imread(normal_root_right  + '/' + 'chessboard0.jpg')

size=(framesize.shape[1],framesize.shape[0])
fourcc = cv2.VideoWriter_fourcc(*'MJPG')
videoWriter = cv2.VideoWriter(r'/Users/Administrator/Desktop/research/video stitching/video/normal video right 2024.avi',fourcc,fps,size,True)

all_img_file = os.listdir(normal_root_right)
all_img_file.sort(key= lambda x:int(x[10:-4]))

for img_file in all_img_file:
    frame = cv2.imread(normal_root_right + '/' + img_file)
    videoWriter.write(frame)
    print(normal_root_right + '/' + img_file + ' done!')
videoWriter.release()

/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard0.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard1.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard2.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard3.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard4.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard5.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard6.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard7.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard8.jpg done!
/Users/Administrator/Desktop/research/video stitching/f

/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard78.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard79.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard80.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard81.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard82.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard83.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard84.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard85.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard86.jpg done!
/Users/Administrator/Desktop/research/video st

/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard155.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard156.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard157.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard158.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard159.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard160.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard161.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard162.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard163.jpg done!
/Users/Administrator/Desktop/research

/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard234.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard235.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard236.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard237.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard238.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard239.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard240.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard241.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard242.jpg done!
/Users/Administrator/Desktop/research

/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard314.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard315.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard316.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard317.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard318.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard319.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard320.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard321.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard322.jpg done!
/Users/Administrator/Desktop/research

/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard394.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard395.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard396.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard397.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard398.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard399.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard400.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard401.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard402.jpg done!
/Users/Administrator/Desktop/research

/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard474.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard475.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard476.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard477.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard478.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard479.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard480.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard481.jpg done!
/Users/Administrator/Desktop/research/video stitching/frames/STR normal right 2024/chessboard482.jpg done!
/Users/Administrator/Desktop/research